# Marketing Funnel Master Table — Cleaning

## What is this notebook?
This notebook cleans `marketing_funnel_master_table.csv` (built by `marketing-funnel.ipynb`, a left
join of `marketing_qualified_leads_dataset.csv` and `closed_deals_dataset.csv` on `mql_id`). Grain:
one row per lead (`mql_id`). A lead is **converted** (`is_converted_flag = True`) if it has a matching
closed deal.

## Two different kinds of missing data on this table
1. **Conversion-gated fields** — `business_segment`, `sdr_id`, `sr_id`, `won_date`, etc. are `NaN`
   for every lead that never converted, by construction of the left join. That's expected and is
   *not* something to clean: it means "never became a deal", not "bad data".
2. **Genuine data-quality issues inside the deals themselves** — found by checking the raw
   `closed_deals_dataset.csv` on its own 842 rows (i.e. only among leads that DID convert):
   - `declared_monthly_revenue` is exactly **0 for ~95% of converted deals**, uniformly across every
     `lead_type` segment. A real business declaring literally R$0/month at a near-constant ~95% rate
     regardless of segment is not plausible — this is almost certainly an unanswered survey field
     encoded as 0, silently mixed in with genuine (often large) declared values. Treating these as
     real zeros would badly distort any revenue analysis.
   - `has_company` / `has_gtin` load as `object` dtype (mixed `True` / `False` / `NaN` strings)
     instead of proper booleans.
   - `lead_behaviour_profile` mixes single categorical values (`cat`, `wolf`, `eagle`, `shark`) with
     comma-joined multi-label values (`cat, wolf`) — inconsistent encoding for grouping/analysis.
   - `time_to_win_days` has 34 deals at exactly 0 (same-day close — plausible, but worth flagging
     rather than silently blending with the rest) and 1 deal at **-2** (won 2 days before first
     contact — an upstream data-entry error).

This notebook fixes what can be fixed, and — matching the approach used for the seller table —
never fabricates a value: where something can't be confidently determined, we add an explicit flag
column instead of imputing or dropping rows.


In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("../data-visualization/marketing_funnel_master_table.csv")
df

,mql_id,first_contact_date,landing_page_id,origin,seller_id,sdr_id,sr_id,won_date,business_segment,lead_type,...,has_gtin,average_stock,business_type,declared_product_catalog_size,declared_monthly_revenue,is_converted_flag,time_to_win_days,lead_age_days,days_to_first_contact_missing_flag,won_date_missing_flag
0,dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,3111,False,True
1,8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,3215,False,True
2,b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,3062,False,True
3,6be030b81c75970747525b843c1ef4f8,2018-01-22,d45d558f0daeecf3cccdffe3c59684aa,email,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,3121,False,True
4,5420aad7fec3549a85876ba1c529bd84,2018-02-21,b48ec5f3b04e9068441002a19df93c6c,organic_search,2c43fb513632d29b3b58df74816f1b06,a8387c01a09e99ce014107505b92388c,4ef15afb4b2723d8f3d81e51ec7afefe,2018-02-26 19:58:54,pet,online_medium,...,NaN,NaN,reseller,NaN,0.0,True,5.0,3091,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,feaba3ffcd2ff97501696c7f9a42f41c,2018-05-22,e42a14209c69c3e9cc6b042620465f12,paid_search,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,3001,False,True
7996,a79cb53cd009ab92e0143b92baa2407b,2018-03-27,c494978688ccf66ad9fad3d6a3338c22,paid_search,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,3057,False,True
7997,68f049a23ab109c6a0f6989bb9a02994,2017-08-27,b48ec5f3b04e9068441002a19df93c6c,organic_search,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,3269,False,True
7998,4f8c96e2509b984329044c6682c88ee9,2017-10-06,a56671a54260a44923d32c2f08fad39c,organic_search,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,3229,False,True


In [3]:
df.shape

(8000, 22)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 22 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   mql_id                              8000 non-null   object 
 1   first_contact_date                  8000 non-null   object 
 2   landing_page_id                     8000 non-null   object 
 3   origin                              7940 non-null   object 
 4   seller_id                           842 non-null    object 
 5   sdr_id                              842 non-null    object 
 6   sr_id                               842 non-null    object 
 7   won_date                            842 non-null    object 
 8   business_segment                    841 non-null    object 
 9   lead_type                           836 non-null    object 
 10  lead_behaviour_profile              665 non-null    object 
 11  has_company                         63 non-

In [5]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
mql_id,8000,8000,dac32acd4db4c29c230538b72f8dd87d,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
first_contact_date,8000,336,2018-05-02,93,NaN,NaN,NaN,NaN,NaN,NaN,NaN
landing_page_id,8000,495,b76ef37428e6799c421989521c0e5077,912,NaN,NaN,NaN,NaN,NaN,NaN,NaN
origin,7940,10,organic_search,2296,NaN,NaN,NaN,NaN,NaN,NaN,NaN
seller_id,842,842,2c43fb513632d29b3b58df74816f1b06,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sdr_id,842,32,4b339f9567d060bcea4f5136b9f5949e,140,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sr_id,842,22,4ef15afb4b2723d8f3d81e51ec7afefe,133,NaN,NaN,NaN,NaN,NaN,NaN,NaN
won_date,842,824,2018-05-04 03:00:00,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
business_segment,841,33,home_decor,105,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lead_type,836,8,online_medium,332,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df.nunique().sort_values()

days_to_first_contact_missing_flag       1
won_date_missing_flag                    2
is_converted_flag                        2
has_gtin                                 2
has_company                              2
business_type                            3
average_stock                            6
lead_type                                8
lead_behaviour_profile                   9
origin                                  10
sr_id                                   22
declared_monthly_revenue                27
sdr_id                                  32
declared_product_catalog_size           33
business_segment                        33
time_to_win_days                       193
first_contact_date                     336
lead_age_days                          336
landing_page_id                        495
won_date                               824
seller_id                              842
mql_id                                8000
dtype: int64

In [7]:
missing = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
)

missing_pct = (
    df.isna()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
})

missing_summary

,missing_count,missing_pct
has_company,7937,99.2125
has_gtin,7936,99.2000
average_stock,7934,99.1750
declared_product_catalog_size,7931,99.1375
lead_behaviour_profile,7335,91.6875
business_type,7168,89.6000
lead_type,7164,89.5500
business_segment,7159,89.4875
time_to_win_days,7158,89.4750
won_date,7158,89.4750


In [8]:
df["mql_id"].duplicated().sum()

0

## Understanding the missing values
Most of the missing-value list above is explained entirely by conversion status: any deal-side
column is `NaN` whenever `is_converted_flag` is `False`. The cell below confirms this is uniform —
if it were, cleaning those columns would mean nothing more than "this lead didn't convert".

In [9]:
deal_cols = [
    "seller_id", "sdr_id", "sr_id", "won_date", "business_segment", "lead_type",
    "lead_behaviour_profile", "has_company", "has_gtin", "average_stock", "business_type",
    "declared_product_catalog_size", "declared_monthly_revenue"
]

# For every deal-side column, missing should line up exactly with "not converted"
for col in deal_cols:
    mismatch = (df[col].isna() != ~df["is_converted_flag"]).sum()
    print(f"{col:35s} mismatch rows: {mismatch}")

seller_id                           mismatch rows: 0
sdr_id                              mismatch rows: 0
sr_id                               mismatch rows: 0
won_date                            mismatch rows: 0
business_segment                    mismatch rows: 1
lead_type                           mismatch rows: 6
lead_behaviour_profile              mismatch rows: 177
has_company                         mismatch rows: 779
has_gtin                            mismatch rows: 778
average_stock                       mismatch rows: 776
business_type                       mismatch rows: 10
declared_product_catalog_size       mismatch rows: 773
declared_monthly_revenue            mismatch rows: 0


Every deal-side column is `NaN` exactly when `is_converted_flag` is `False` — confirming those
nulls are structural, not data-quality problems. So we only need to clean **within the 842 converted
rows**, where the fields above should always be present but sometimes still hide real issues.

In [10]:
won = df[df["is_converted_flag"]]
print("Converted deals:", len(won))
won[["business_segment", "lead_type", "lead_behaviour_profile", "has_company", "has_gtin",
     "average_stock", "business_type", "declared_product_catalog_size",
     "declared_monthly_revenue"]].isna().sum()

Converted deals: 842


business_segment                   1
lead_type                          6
lead_behaviour_profile           177
has_company                      779
has_gtin                         778
average_stock                    776
business_type                     10
declared_product_catalog_size    773
declared_monthly_revenue           0
dtype: int64

Even among the 842 converted deals, several fields are still mostly missing
(`has_company`, `has_gtin`, `average_stock`, `declared_product_catalog_size` are ~92-93% empty) —
these read as an optional onboarding survey that most sales reps never filled in, not something we
can impute. We leave them as `NaN` (no fabricated values) but make sure the dtypes are correct.

In [11]:
df["business_segment"].value_counts().head(10)

business_segment
home_decor                         105
health_beauty                       93
car_accessories                     77
household_utilities                 71
construction_tools_house_garden     69
audio_video_electronics             64
computers                           34
pet                                 30
food_supplement                     28
food_drink                          26
Name: count, dtype: int64

In [12]:
df["has_company"].value_counts(dropna=False)

has_company
NaN      7937
True       58
False       5
Name: count, dtype: int64

In [13]:
df["declared_monthly_revenue"].describe()

count    8.420000e+02
mean     7.337768e+04
std      1.744799e+06
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      5.000000e+07
Name: declared_monthly_revenue, dtype: float64

## The declared_monthly_revenue problem
`declared_monthly_revenue` is 0 for the vast majority of converted deals. If this were a genuine
business signal, the zero-rate should vary meaningfully by lead type (e.g. brand-new online sellers
declaring 0 makes sense, established `industry` sellers declaring 0 does not). Checking below.

In [14]:
zero_rate_by_type = (
    won.assign(is_zero=won["declared_monthly_revenue"].eq(0))
       .groupby("lead_type")["is_zero"]
       .mean()
       .sort_values(ascending=False)
)
zero_rate_by_type

lead_type
online_top         1.000000
online_beginner    0.964912
industry           0.959350
offline            0.951923
online_medium      0.948795
online_big         0.936508
online_small       0.935065
other              0.000000
Name: is_zero, dtype: float64

The zero-rate sits at ~93-100% across every single lead type, including `industry` and
`online_top` sellers who would be expected to report real revenue. A near-constant ~95% zero rate
regardless of business maturity is the signature of an unanswered form field defaulting to 0, not a
real declared value. We therefore treat `declared_monthly_revenue == 0` as **unknown**, not zero:
convert it to `NaN` and add a `has_declared_revenue` flag so downstream analysis can tell "declared
$0" (none, after this cleaning) apart from "didn't declare" (flagged) without silently averaging fake
zeros into revenue statistics. The same logic applies to `declared_product_catalog_size`, which is
naturally paired with the revenue question on the same survey.

In [15]:
df[df["declared_product_catalog_size"] == 0][
    ["mql_id", "declared_product_catalog_size", "declared_monthly_revenue"]
].shape[0]

0

## Cleaning decisions
- `declared_monthly_revenue` / `declared_product_catalog_size`: 0 is not a plausible real answer at
  this frequency — recode `0` to `NaN` and add `has_declared_revenue` / `has_declared_catalog_size`
  flags so a missing declaration is never confused with an actual $0 or empty catalog.
- `has_company` / `has_gtin`: cast the `True`/`False`/`NaN` strings to a proper nullable boolean
  dtype instead of `object`.
- `lead_behaviour_profile`: split into a `lead_behaviour_profile` (primary/first label) and a
  `multi_behaviour_flag` (True if the lead was tagged with more than one behaviour) — only 16 of 665
  labeled deals have more than one tag, so collapsing to a primary label keeps the column usable for
  simple grouping while the flag preserves the "mixed behaviour" information.
- `time_to_win_days`: flag (don't drop) the 34 same-day closes and the 1 negative value via
  `time_to_win_valid_flag`, so time-to-win statistics can exclude the one clear data error
  (`time_to_win_days < 0`) while keeping same-day closes, which are plausible.
- Dates: ensure `first_contact_date` / `won_date` are proper datetimes.
- No duplicate `mql_id`s existed, so no deduplication was needed.

In [16]:
new_df = df.copy()

new_df["has_declared_revenue"] = new_df["declared_monthly_revenue"].notna() & new_df["declared_monthly_revenue"].ne(0)
new_df.loc[new_df["declared_monthly_revenue"] == 0, "declared_monthly_revenue"] = np.nan

new_df["has_declared_catalog_size"] = new_df["declared_product_catalog_size"].notna() & new_df["declared_product_catalog_size"].ne(0)
new_df.loc[new_df["declared_product_catalog_size"] == 0, "declared_product_catalog_size"] = np.nan

In [17]:
for col in ["has_company", "has_gtin"]:
    new_df[col] = new_df[col].map({"True": True, "False": False, True: True, False: False}).astype("boolean")

new_df[["has_company", "has_gtin"]].dtypes

has_company    boolean
has_gtin       boolean
dtype: object

In [18]:
new_df["multi_behaviour_flag"] = new_df["lead_behaviour_profile"].str.contains(",", na=False)
new_df["lead_behaviour_profile"] = new_df["lead_behaviour_profile"].str.split(",").str[0].str.strip()

new_df["lead_behaviour_profile"].value_counts(dropna=False)

lead_behaviour_profile
NaN      7335
cat       415
eagle     129
wolf       95
shark      26
Name: count, dtype: int64

In [19]:
new_df["time_to_win_valid_flag"] = new_df["time_to_win_days"].isna() | new_df["time_to_win_days"].ge(0)

new_df.loc[~new_df["time_to_win_valid_flag"],
           ["mql_id", "first_contact_date", "won_date", "time_to_win_days"]]

,mql_id,first_contact_date,won_date,time_to_win_days
6357,b91cf8812365f50ff4bda4bcd6206b05,2018-03-08,2018-03-06 19:38:55,-2.0


In [20]:
date_cols = ["first_contact_date", "won_date"]

for col in date_cols:
    new_df[col] = pd.to_datetime(new_df[col], errors="coerce")

In [21]:
new_df.isna().sum()

mql_id                                   0
first_contact_date                       0
landing_page_id                          0
origin                                  60
seller_id                             7158
sdr_id                                7158
sr_id                                 7158
won_date                              7158
business_segment                      7159
lead_type                             7164
lead_behaviour_profile                7335
has_company                           7937
has_gtin                              7936
average_stock                         7934
business_type                         7168
declared_product_catalog_size         7931
declared_monthly_revenue              7955
is_converted_flag                        0
time_to_win_days                      7158
lead_age_days                            0
days_to_first_contact_missing_flag       0
won_date_missing_flag                    0
has_declared_revenue                     0
has_declare

In [22]:
new_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 26 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   mql_id                              8000 non-null   object        
 1   first_contact_date                  8000 non-null   datetime64[ns]
 2   landing_page_id                     8000 non-null   object        
 3   origin                              7940 non-null   object        
 4   seller_id                           842 non-null    object        
 5   sdr_id                              842 non-null    object        
 6   sr_id                               842 non-null    object        
 7   won_date                            842 non-null    datetime64[ns]
 8   business_segment                    841 non-null    object        
 9   lead_type                           836 non-null    object        
 10  lead_behaviour_profile  

## Before saving — what changed vs. the raw file
- Rows: **8,000 -> 8,000** (no leads dropped).
- Columns: **22 -> 26** (added `has_declared_revenue`, `has_declared_catalog_size`,
  `multi_behaviour_flag`, `time_to_win_valid_flag`).
- `declared_monthly_revenue` / `declared_product_catalog_size`: `0` values (the ~95% "unanswered"
  placeholder) recoded to `NaN`, now distinguishable via the two new `has_*` flags.
- `has_company` / `has_gtin`: now nullable boolean dtype instead of `object`.
- `lead_behaviour_profile`: collapsed to a single primary label; the 16 multi-label rows are marked
  via `multi_behaviour_flag` instead of being silently truncated with no trace.
- `time_to_win_days`: unchanged values, but the 1 negative (invalid) row is now flagged via
  `time_to_win_valid_flag = False` instead of being silently included in any downstream average.
- `first_contact_date` / `won_date`: proper `datetime64` instead of strings.
- No duplicate `mql_id`s existed, so no deduplication was needed.

In [23]:
total_leads = new_df["mql_id"].nunique()
conversion_rate = new_df["is_converted_flag"].mean()
declared_revenue_coverage = new_df["has_declared_revenue"].sum()
invalid_time_to_win = (~new_df["time_to_win_valid_flag"]).sum()

print("Total Leads:", total_leads)
print("Conversion Rate:", conversion_rate)
print("Converted deals with a usable declared revenue:", declared_revenue_coverage)
print("Deals with invalid (negative) time_to_win_days:", invalid_time_to_win)

Total Leads: 8000
Conversion Rate: 0.10525
Converted deals with a usable declared revenue: 45
Deals with invalid (negative) time_to_win_days: 1


In [24]:
new_df.to_csv("../data-visualization/cleaned_marketing_funnel_master_table.csv", index=False)

In [25]:
print("Shape of cleaned marketing funnel table:", new_df.shape)
print("Saved as: ../data-visualization/cleaned_marketing_funnel_master_table.csv")

Shape of cleaned marketing funnel table: (8000, 26)
Saved as: ../data-visualization/cleaned_marketing_funnel_master_table.csv
